### Bidirektionale Breitensuche
Hat jeder Knoten viele Nachbarn,
wächste die Zahl der Knoten exponentiell mit dem Abstand von Start-Knoten.
Suchen wir z.B. ein Ziel mit Abstand 10, sind sehr viel mehr Knoten zu besuchen, als 
bei einem Ziel mit Abstand 5.
Deshalb lohnt es sich, die Erkundungen vom Start und Ziel aus zu starten.
Haben Start und Ziel Abstand 10, dann treffen sich die Suchfronten bereits nach 5 Schritten.

Die gleichzeitige Breitensuche von Start und Ziel aus nennt man **bidirektionale Breitensuche**. 
Sie ist anwendbar, falls der Zielknoten bekannt ist und der Suchraum rückwärts durchlaufen werden kann.

Der Suchraum wächst exponentiell mit der Tiefe.
Hat z.B. der Zielknoten Abstand 10 vom Start und
hat jeder  Knoten im Durchschnitt 6 Kinder,
sind  ca. 6^10 = 60_466_176 Knoten zu besuchen, um bis in Tiefe 10 vorzudringen.
Wird die Suche vom Start und vom Ziel her gestartet, 
treffen sich die Suchfronten in der Mitte.
Es sind also nur ca. 2*6^5 = 15_552 Knoten zu besuchen.

<img src='/files/images/Bidirektionales_Suchen.png'>

### Beispiel
Wir suchen in einen Graphen mit 100 Knoten, wo jeder Knoten 4 Nachbarn hat.
Im Allgemeinen ergibt dies einen sehr grosser Suchraum.
Im vorliegenden Fall (`graph.py`) sind die Nachbarn sehr speziell gewählt:

- Die Knoten 1 bis 100 sind auf einem Kreis angeordnet.
- Die 4 Nachbarn eines Knoten sind die beiden Knoten vor und hinter ihm.

<img src='/files/images/Ringgitter.png' width='300px'>

In [ ]:
from graph import graph
import search_strategies as S


def get_neighbors(n):
    return graph[n]


def is_path(path, get_neighbors):
    for i in range(len(path)-1):
        if path[i+1] not in get_neighbors(path[i]):
            return False
    return True

In [ ]:
# Welche Knoten haben welchen Abstand vom Knoten 1?
start = 1
goal = None
node, go_back, dist_dict = S.search_bf(start, get_neighbors, goal)
path = S.get_path_home(node, go_back)
print(f'Length of path: {len(path)}')
print(f'Distance to goal: {dist_dict.get(goal)}')
path[:10]

In [ ]:
dist_dict.get(node)

In [ ]:
dist_node = {}
for node, dist in dist_dict.items():
    if dist not in dist_node:
        dist_node[dist] = [node]
    else:
        dist_node[dist].append(node)

dist_node = dict(sorted(dist_node.items()))
dist_node

In [ ]:
# Unidirektionale BFS
start = 1
goal = 58
node, go_back, dist_dict = S.search_bf(start, get_neighbors, goal)
path = S.get_path_home(node, go_back)
print(f'Length of path: {len(path)}')
print(f'Distance to goal: {dist_dict.get(goal)}')
path[:10]

In [ ]:
# Bidirektionale BFS
start = 1
goal = 58

midpoint, go_backs = S.search_bibf(start, get_neighbors, goal)
path = S.join_paths_home(midpoint, go_backs)
len(path), path[:10]

In [ ]:
is_path(path, get_neighbors)

### Aufgabe
1. Modifiziere die BFS-Suche so, dass alle kreisfreien Pfade von 1 nach 10 der Länge 6 gefunden werden.
   Teste, für welche n noch kürzeste Pfade von 1 nach n (schnell) gefunden werden.  
   Ermittle die Länge des kürzesten Pfades mif BFS.

   **Hinweis**: Statt Knoten werden nun Pfade gesucht, also Tuple von Knoten der Form
   (1, 2, 4, 6). Hier brauchen wir keinen `go_back` Dict, es genügt, alle
   gesehenen Pfade in eine Menge `seen` aufzunehmen.
   Bevor ein Pfad um einen Knoten n verlängert wird, teste, ob n noch nicht im Pfad vorkommt.

   
   
1. Verwende nun folgende Heuristik, um mit einer greedy Suche alle kurzen Pfade zu finden.
   Setze eine Variable `current_dist = 1000`. Suche solange nach Pfaden, bis die Pfadlänge wieder zunimmt.

   
```python
def h(path, goal):
    '''geschaetzte Gesamtlaenge des Pfades'''
    return len(path) + (abs(path[-1] - goal)+1) // 2
```

In [ ]:
def h(path, goal):
    '''geschaetzte Gesamtlaenge des Pfades'''
    return len(path) + (abs(path[-1] - goal)+1) // 2

In [ ]:
path = (1, 2)
for goal in (3, 4, 5, 100, 99, 98):
    length = h(path, goal)
    print(f'geschaetzte Gesamtlaenge bis {goal}: {length}')

In [ ]:
def get_min_len(start, goal):
    '''liefert die Laenge des kuerzesten Pfades von start nach goal '''
    _, _, dist_dict = S.search_bf(start, get_neighbors, goal)
    length = dist_dict[goal]
    return length

In [ ]:
get_min_len(1, 52)

***
Pfade von 1 nach 10 der Länge kleiner gleich 10 direkt mit `S.search_bf` finden:  
- start = `(1,)`
- brauchen Funktion, die alle Verlängerungen mit neuen Knoten eines Pfades liefert
***

In [ ]:
def get_path_extensions(path):
    nodes = set(path)
    for node in graph[path[-1]]:
        if node not in nodes:
            yield path + (node,)

In [ ]:
start = (1,)
goal = 10
path, go_back, dist_dict = S.search_bf(start, get_path_extensions, None, max_depth=10)

In [ ]:
paths = [path for path in dist_dict if path[-1] == goal]
len(paths)

In [ ]:
paths[:2] + [paths[-2:]]

In [ ]:
paths = [path for path, n in dist_dict.items() if path[-1] == goal and n == 6 - 1]
len(paths)

***
Pfade mit modifizierter Variante von `S.search_bf` finden.
***

In [ ]:
from collections import deque


def search_bf_paths(node, get_neighbors, goal, max_len=None):
    count = 0
    path = (node,)
    paths_to_visit = deque([path])
    seen = {path}

    while paths_to_visit:
        count += 1
        path = paths_to_visit.pop()
        if path[-1] == goal:
            yield path

        if max_len and len(path) == max_len:
            continue

        nodes = set(path)
        for node in get_neighbors(path[-1]):
            if node in nodes:
                continue

            new_path = path + (node,)
            if new_path in seen:
                continue
            seen.add(new_path)
            paths_to_visit.appendleft(new_path)

    print(f'Failure. Count: {count}')

In [ ]:
start = 1
goal = 10
paths_gen = search_bf_paths(start, get_neighbors, goal, max_len=6)
paths = list(paths_gen)
len(paths)

In [ ]:
paths = [path for path in paths if len(path) == 6]
len(paths)

In [ ]:
paths

***
Kürzeste Pfade mit geeigneter Heuristik `h` finden.

In `current_len` wird die Länge des zuletzt gefundenen Pfades gespeicher.
Zu Beginn ist dieser Wert zu gross.

Sobald wir einen kürzesten Pfad gefunden haben, wird dessen Länge in
`current_len` gespeichert. Die Suche stoppt, falls
ein neuer Pfad länger als  `current_len` ist.
***

In [ ]:
import heapq


def search_greedy_paths(node, get_neighbors, h, goal):
    count = 0
    path = (node,)
    priority = (h(path), count)
    paths_to_visit = [(priority, path)]
    seen = {path}
    current_len = 1000

    while paths_to_visit:
        _, path = heapq.heappop(paths_to_visit)
        if path[-1] == goal:
            n = len(path)
            if current_len < n:
                return

            current_len = n
            yield path

        nodes = set(path)
        for node in get_neighbors(path[-1]):
            if node in nodes:
                continue

            new_path = path + (node,)
            if new_path in seen:
                continue

            seen.add(new_path)
            count += 1
            priority = (h(new_path), count)
            heapq.heappush(paths_to_visit, (priority, new_path))

    print(f'Failure. Count: {count}')
    return None, go_back

In [ ]:
start = 1  # 3 liefer bereits zu lange Pfade
goal = 70
paths_gen = search_greedy_paths(start, get_neighbors, lambda p: h(p, goal), goal)
paths = list(paths_gen)
len(paths), set(len(path) for path in paths)

In [ ]:
paths

In [ ]:
start = 4
goal = 99
paths_gen = search_greedy_paths(start, get_neighbors, lambda p: h(p, goal), goal)
paths = list(paths_gen)
len(paths), set(len(path) for path in paths)

In [ ]:
paths[0][:5]

***
Bessere Heuristik
```python
def h(path, goal):
    '''konservativ geschaetzte Gesamtlaenge des Pfades'''
    start = path[-1]
    if start > goal:
        start, goal = goal, start

    d1 = goal - start
    d2 = start + 100 - goal

    return len(path) + min(d1, d2) // 2
```
***

In [ ]:
def h(path, goal):
    '''konservativ geschaetzte Gesamtlaenge des Pfades'''
    start = path[-1]
    if start > goal:
        start, goal = goal, start

    d1 = goal - start
    d2 = start + 100 - goal

    return len(path) + min(d1, d2) // 2

In [ ]:
path = (1, 2)
for goal in (3, 4, 5, 100, 99, 98):
    length = h(path, goal)
    print(f'konservativ geschaetzte Gesamtlaenge bis {goal}: {length}')